# RITINI: Inferring Dynamic Regulatory Interaction Graphs from Time Series Data with Perturbations 

Prerequisites:
- Trained MIOFlow and decoded trajectories back to gene space.

In this notebook we will:
- Run RITINI to infer gene dynamics in gene regulatory networks

# Import libraries, set path and device

In [ ]:
# Standard library imports
import warnings

import dgl
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import scanpy as sc
import torch

from torch.optim.lr_scheduler import StepLR

# Local application imports
from omics_toolbox.gode.utils import get_device
from omics_toolbox.gode.data import make_train_test_dataframe
from omics_toolbox.ritini_module import ritini

# Suppress specific warnings
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    message=".*unique with argument that is not not a Series.*"
)

DGL backend not selected or invalid.  Assuming PyTorch for now.


Setting the default backend to "pytorch". You can change it in the ~/.dgl/config.json file or export the DGLBACKEND environment variable.  Valid options are: pytorch, mxnet, tensorflow (all lowercase)


In [2]:
device = get_device()

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

seed = 3
torch.manual_seed(seed)
np.random.seed(seed)

# Load and Preprocess Dataset

Load the dataset (output from MIOFlow) of shape ((n_timepoints, n_trajectrories, n_genes))

In [3]:
"""
Load MIOFlow inferred trajectories.
Shape (n_timepoints, n_trajectrories, n_genes)
"""
# trajectories = np.load(f"/vast/palmer/pi/krishnaswamy_smita/hcd22/nikjoshi_new_2/results/dimchanger/dimchanger_{subset_name}/{flow_name}_gene_sp.npy", allow_pickle=True)
trajectories = np.load('../../results/scRNAseq/trajectories_gene_space.npy', allow_pickle=True)
"""
Load gene names for plotting purposes
Shape (n_genes,)
"""
# genes = np.load(f'/vast/palmer/pi/krishnaswamy_smita/hcd22/nikjoshi_new_2/results/granger/granger_{subset_name}/{flow_name}_gene_names.npy', allow_pickle=True)
adata = sc.read('../../data/processed/adata_mioflow.h5ad')
genes = adata.var_names
genes = genes.str.replace(r'\s*\(ENSG[^\)]*\)', '', regex=True)
"""
Annotations are clusters of trajectories for a specific lineage. Here we assume just one set of trajectories. 
All labeled as 0.
"""
annotations = np.zeros(trajectories.shape[0])

"""
For simplicity, we will just train on the mean trajectory.
"""
mean_trajectories = trajectories.mean(axis=1, keepdims=True)

traj_data = {
    'trajectories': mean_trajectories,
    'genes': genes, 
    'annotations': annotations
}

In [4]:
"""
Load Granger causality data to construct the prior graph.
"""
granger_df_all_T = pd.read_csv('../../data/processed/out_traj_p_250.csv', index_col=0)
epsilon = 1e-10
threshold = 5  # You can adjust this value as needed

neg_log_p = -np.log(granger_df_all_T + epsilon)
granger_df_all_T = neg_log_p.where(neg_log_p > threshold, 0)

preds = granger_df_all_T.to_numpy()
tf_count = preds.shape[0]

# Creating Prior Graph
- Each node is a gene and the edges is the interaction between the genes.

In [5]:
# --- Set up labels (gene names) ---
top_genes = list(granger_df_all_T.columns)  # output genes
in_genes = list(granger_df_all_T.index)     # input genes (TFs)

# --- Create directed NetworkX graph ---
G = nx.DiGraph()

# Add nodes (we'll use all output genes as node set)
for idx, gene in enumerate(top_genes):
    G.add_node(idx, label=gene)

# Add edges: r (input gene) -> c (output gene) if significant
for i, input_gene in enumerate(in_genes):
    for j, output_gene in enumerate(top_genes):
        if preds[i, j] >= threshold:
            G.add_edge(i, j)  # node indices match column indices of `top_genes`

# --- Convert NetworkX to DGL ---
edges = list(G.edges())
if len(edges) > 0:
    u, v = np.array(edges).T
    u = torch.tensor(u, dtype=torch.int32)
    v = torch.tensor(v, dtype=torch.int32)
else:
    u = v = torch.tensor([], dtype=torch.int32)

g = dgl.graph((u, v))

# Add dummy node features (e.g., 1D ones)
g.ndata['feat'] = torch.ones((g.number_of_nodes(), 1))

# --- For plotting (optional) ---
ref_g = g.to_networkx()
ref_pos = nx.spring_layout(ref_g.to_undirected(), seed=seed)

# Color and label
for idx, node in enumerate(ref_g.nodes()):
    ref_g.nodes[node]['color'] = plt.get_cmap('viridis', len(top_genes))(idx)
    ref_g.nodes[node]['label'] = top_genes[idx]

In [6]:
""" 
Create the adjacency matrix.
"""
adjacency_matrix = nx.adjacency_matrix(G)
adjacency_matrix_negative = 1 - adjacency_matrix.todense() - np.eye(g.number_of_nodes())

In [7]:
"""
Split edges into training and testing sets for link prediction loss.
"""
# Mapping for edge ids
edge_ids = np.arange(g.number_of_edges())

# Shuffle
edge_ids = np.random.permutation(edge_ids)

test_size_percent = 30
test_size_fraction = test_size_percent / 100

edge_test_size = int(len(edge_ids) * test_size_fraction)
edge_train_size = g.number_of_edges() - edge_test_size

edge_test_pos_u = u[edge_ids[:edge_test_size]]
edge_test_pos_v = v[edge_ids[:edge_test_size]]

edge_train_pos_u = u[edge_ids[edge_test_size:]]
edge_train_pos_v = v[edge_ids[edge_test_size:]]

neg_u, neg_v = np.where(adjacency_matrix_negative != 0)
neg_edge_ids = np.random.choice(len(neg_u), g.number_of_edges())

edge_test_neg_u = neg_u[neg_edge_ids[:edge_test_size]]
edge_test_neg_v = neg_v[neg_edge_ids[:edge_test_size]]

edge_train_neg_u = neg_u[neg_edge_ids[edge_test_size:]]
edge_train_neg_v = neg_v[neg_edge_ids[edge_test_size:]]

# Train RiTINI
- Saves plots of the ground truth dynamics vs predicted dynamics
- Saves the inferred graph (gene regulatory network)

In [8]:
import re

# Extract the part before the space (i.e., the gene symbol)
clean_top_genes = [re.sub(r'\s+\(.*?\)_y$', '', h) for h in top_genes]
top_genes = clean_top_genes

In [9]:
seen = set()
gene_subset_indices = []
for i, k in enumerate(traj_data['genes']):
    if k in top_genes and k not in seen:
        seen.add(k)
        gene_subset_indices.append(i)
gene_subset_indices = np.array(gene_subset_indices)

cell_subset_indices = np.random.choice(traj_data['trajectories'].shape[1], traj_data['trajectories'].shape[1], replace=False)

In [10]:
gene_subset_indices.shape

(250,)

In [11]:
trajs = traj_data['trajectories']
trajs = trajs[::3]
trajs = trajs[:, cell_subset_indices]
trajs = trajs[:, :, gene_subset_indices]
traj_f = trajs.reshape(-1, trajs.shape[2])

In [12]:
pseudotimes = np.linspace(0, 1, trajs.shape[0])

In [13]:
annot_repeated = np.repeat(traj_data['annotations'][cell_subset_indices], trajs.shape[0])
pt_repeated = np.tile(pseudotimes, trajs.shape[1])
df = pd.DataFrame(traj_f, columns=top_genes, index=[f'cell_{i}' for i in range(traj_f.shape[0])])
df['pseudotime'] = pt_repeated

df['cell_types'] = [f'cell_type_{a}' for a in annot_repeated]
num_cell_types = len(df['cell_types'].unique())

In [14]:
df_train, df_test = make_train_test_dataframe(df)

In [15]:
n_cells_at_t = df['pseudotime'].value_counts()[0]

time_bins = np.sort(df.pseudotime.unique())
cell_types = np.sort(df.cell_types.unique())

t0, *_, tn = time_bins
time_tensor = torch.Tensor(time_bins)#.to(device)

in_feats = cell_types.size * n_cells_at_t
out_feats = cell_types.size * n_cells_at_t

In [16]:
# Create RITINI instance first
graph_trainer = ritini.RITINI(g, in_feats, out_feats, device)
# Initialize the model
model = graph_trainer.model
device = 'cpu'
model = model.to(device)
# Call the train_test method on the instance
train_g, train_pos_g, train_neg_g, test_pos_g, test_neg_g = graph_trainer.train_test(
    edge_ids, edge_train_pos_u, edge_train_pos_v, edge_train_neg_u, edge_train_neg_v, 
    edge_test_pos_u, edge_test_pos_v, edge_test_neg_u, edge_test_neg_v, edge_test_size
)

In [17]:
""" 
Hyperparameters for training 
"""
optimizer = torch.optim.AdamW(model.parameters(), lr=0.1, weight_decay=5e-4)
scheduler = StepLR(optimizer, step_size=350, gamma=0.1)
criterion = torch.nn.MSELoss()

steps = 100
verbose_step = 1

lambda_l1 = 10
add_n = 5
del_n = 5
link_step = 2
sample_size = 10

In [18]:
graph_trainer.train_loop(
        model, optimizer, scheduler, criterion, clean_top_genes,
        train_g, train_pos_g, train_neg_g, test_pos_g, test_neg_g, 
        df_train, n_cells_at_t, time_bins, steps, link_step, add_n, del_n,
        verbose_step, num_cell_types, cell_types, ref_pos, ref_g)

[1],	 Loss: 0.00005,	 AUC: 0.50170
[2],	 Loss: 0.00014,	 AUC: 0.47740
[3],	 Loss: 0.00018,	 AUC: 0.47148
[4],	 Loss: 0.00021,	 AUC: 0.46872
[5],	 Loss: 0.00026,	 AUC: 0.47064
[6],	 Loss: 0.00030,	 AUC: 0.47555
[7],	 Loss: 0.00034,	 AUC: 0.47897
[8],	 Loss: 0.00038,	 AUC: 0.48074
[9],	 Loss: 0.00041,	 AUC: 0.48183
[10],	 Loss: 0.00044,	 AUC: 0.48264
[11],	 Loss: 0.00046,	 AUC: 0.48333
[12],	 Loss: 0.00048,	 AUC: 0.48403
[13],	 Loss: 0.00051,	 AUC: 0.48466
[14],	 Loss: 0.00053,	 AUC: 0.48494
[15],	 Loss: 0.00054,	 AUC: 0.48586
[16],	 Loss: 0.00056,	 AUC: 0.48660
[17],	 Loss: 0.00058,	 AUC: 0.48721
[18],	 Loss: 0.00059,	 AUC: 0.48780
[19],	 Loss: 0.00061,	 AUC: 0.48810
[20],	 Loss: 0.00062,	 AUC: 0.48848
[21],	 Loss: 0.00063,	 AUC: 0.48889
[22],	 Loss: 0.00064,	 AUC: 0.48916
[23],	 Loss: 0.00066,	 AUC: 0.48946
[24],	 Loss: 0.00067,	 AUC: 0.48963
[25],	 Loss: 0.00068,	 AUC: 0.48985
[26],	 Loss: 0.00068,	 AUC: 0.49005
[27],	 Loss: 0.00069,	 AUC: 0.49023
[28],	 Loss: 0.00070,	 AUC: 0.49047
[